# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://croissant.mlhub.social/docs/index.html) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant pandas matplotlib seaborn

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"{metadata.name}\n\nDescription: {metadata.description}\n\nCitation: {getattr(metadata, 'citeAs', 'N/A')}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")


## 2. Data Overview
Review available record sets, their fields, and column `@id`s.

**Note**: All references to record sets, fields, and columns use their `@id`.

In [ ]:
from pprint import pprint

# List all record sets and associated information
record_sets = list(dataset.record_sets)
print(f"Record sets found:")
for record_set in record_sets:
    print(f"- @id: {record_set['@id']} (Name: {record_set.get('name', 'N/A')})")
    if 'field' in record_set:
        fields = record_set['field']
        if not isinstance(fields, list):
            fields = [fields]
        print("  Fields/Columns:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - @id: {field.get('@id', '?')} (Name: {field.get('name', '?')})")
            else:
                print(f"    - @id: {field}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We will demonstrate extraction for every available record set by their `@id` (see overview above). If there are multiple record sets, all are loaded as DataFrames.

In [ ]:
# Extract data from each record set into a dictionary of dataframes, using @id keys
dataframes = {}

for record_set in dataset.record_sets:
    record_set_id = record_set['@id']
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {list(df.columns)}\n  Example rows:")
    print(df.head(2))
    print()

# Pick the main record set for further analysis (typically the first one)
if len(dataframes) > 0:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"First record set ID: {first_record_set_id}")
    main_df = dataframes[first_record_set_id]
else:
    raise ValueError('No record sets found in this dataset.')


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's examine numerical and categorical fields as indicated in the dataset. If the protein abundance field is present, we will perform operations on it.

In [ ]:
# Attempt to automatically determine a main numeric field for demonstration
numeric_columns = main_df.select_dtypes(include='number').columns.tolist()
print(f"Numeric fields in main record set: {numeric_columns}")

if len(numeric_columns) > 0:
    numeric_field = numeric_columns[0]  # Use first available numeric column
    print(f"Using numeric field: '{numeric_field}'")

    # Set a threshold based on field statistics
    threshold = main_df[numeric_field].mean() if not main_df[numeric_field].isnull().all() else 0

    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by another (non-numeric) field, e.g., by 'description' if present
    possible_group_fields = [col for col in main_df.columns if main_df[col].dtype == 'object' and col != numeric_field]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        )
        print(f"\nGrouped mean data by '{group_field}':")
        print(grouped_df.head())
else:
    print("No suitable numeric fields found for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_columns) > 0:
    field = numeric_columns[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[field], kde=True, bins=30, color='skyblue')
    plt.title(f'Distribution of {field}')
    plt.xlabel(field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # If a second numeric column exists, make a scatter plot
    if len(numeric_columns) > 1:
        field2 = numeric_columns[1]
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=main_df[field], y=main_df[field2])
        plt.title(f'{field} vs. {field2} scatter')
        plt.xlabel(field)
        plt.ylabel(field2)
        plt.tight_layout()
        plt.show()
else:
    print('No numeric columns found for visualization.')


## 6. Conclusion

- Successfully loaded and inspected the FAIR² Croissant-format proteomics dataset using `mlcroissant`.
- Explored record set structure and demonstrated extraction by unique `@id`.
- Performed basic exploratory data analysis and visualizations on available numeric fields.

For advanced usage, see [mlcroissant API documentation](https://croissant.mlhub.social/docs/index.html) or extend these steps for deeper machine learning workflows!
